# **Prototype # 01**
---
# 1. Training  gating network

In [2]:
# ---------------------------------------------------
# Paths management
# ---------------------------------------------------
import os

# Project path
PROJECT_PATH = os.path.dirname(os.getcwd())

if os.path.exists(PROJECT_PATH):
    print(f"PROJECT_PATH = {PROJECT_PATH}")
else:
    print(f"WARNING! PROJECT_PATH not found! = {PROJECT_PATH}")
    
# Data path
DATA_PATH = PROJECT_PATH + "/data"
if os.path.exists(DATA_PATH):
    print(f"DATA_PATH = {DATA_PATH}")
else:
    print(f"WARNING! DATA_PATH not found! = {DATA_PATH}")

OUTPUT_DIR = DATA_PATH


# ---------------------------------------------------
# Used paths
# ---------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

PROJECT_PATH = /home/mambo/Documents/Projet_fil_rouge/Neural_Network_Forecasting
DATA_PATH = /home/mambo/Documents/Projet_fil_rouge/Neural_Network_Forecasting/data


In [3]:
# ---------------------------------------------------
# Load dataset
# ---------------------------------------------------
df_merged = pd.read_csv(DATA_PATH + "/experts_and_features_for_moe_nn_01.csv")
df_merged.info()
#df_merged.head()

# ---------------------------------------------------
# DATA LOADER
# ---------------------------------------------------
#**TO DO**

<class 'pandas.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          7000 non-null   int64  
 1   Date_Heure          7000 non-null   str    
 2   y_true              7000 non-null   float64
 3   randomforest        7000 non-null   float64
 4   lgbm                7000 non-null   float64
 5   elasticnet          7000 non-null   float64
 6   Wind_Norm           7000 non-null   float64
 7   Wind_Norm_Cubes     7000 non-null   float64
 8   wind_cv_3h          7000 non-null   float64
 9   Wind_Dir_Meteo_sin  7000 non-null   float64
 10  Wind_Dir_Meteo_cos  7000 non-null   float64
 11  Air_density         7000 non-null   float64
 12  Hour_sin            7000 non-null   float64
 13  Hour_cos            7000 non-null   float64
 14  Month_sin           7000 non-null   float64
 15  Month_cos           7000 non-null   float64
dtypes: float64(14), i

In [5]:
import torch

# ---------------------------------------------------
# Data split
# ---------------------------------------------------
n = len(df_merged)

train_end = int(0.7 * n)   # 4900 (70%)
val_end   = int(0.85 * n)  # 5950 (15%)

train_df = df_merged.iloc[:train_end]
val_df   = df_merged.iloc[train_end:val_end]
test_df  = df_merged.iloc[val_end:]

#   [--------- TRAIN --------][--- VAL ---][--- TEST ---]
#   0                     4900         5950          7000


"""
gating_features = [
    "Wind_Norm",
    "Wind_Norm_Cubes", 
    "wind_cv_3h",
    "Wind_Dir_Meteo_sin",
    "Wind_Dir_Meteo_cos",
    "Air_density",
    "Hour_sin",
    "Hour_cos",
    "Month_sin",
    "Month_cos"
]"""


expert_features = [
    "randomforest",
    "lgbm",
    "elasticnet"]

gating_features = [
    "Wind_Norm"]


X_train = train_df[gating_features]
X_val = val_df[gating_features]
X_test  = test_df[gating_features]

E_train = train_df[expert_features]
E_val = val_df[expert_features]
E_test  = test_df[expert_features]

y_train = train_df["y_true"]
y_val = val_df["y_true"]
y_test  = test_df["y_true"]


# ---------------------------------------------------
# Normalization (on training features ONLY)
# ---------------------------------------------------
"""ating_features = [
    "Wind_Norm",
    "Wind_Norm_Cubes",
    "wind_cv_3h",
    "Wind_Dir_Meteo_sin",
    "Wind_Dir_Meteo_cos",
    "Air_density",
    "Hour_sin",
    "Hour_cos",
    "Month_sin",
    "Month_cos"
]
        TRAIN         VALIDATION        TEST
          ↓               ↓              ↓
      fit scaler      transform       transform
          ↓               ↓              ↓
      mean/std       same mean/std   same mean/std
"""

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# We learn statistics (mean/std) on training data
X_train_scaled = scaler.fit_transform(X_train)
# These statistics are applied then to validation and test datasets
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# ---------------------------------------------------
# Convert into PyTorch tensors
# ---------------------------------------------------
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)

E_train_t = torch.tensor(E_train.to_numpy(), dtype=torch.float32)
E_val_t = torch.tensor(E_val.to_numpy(), dtype=torch.float32)
E_test_t = torch.tensor(E_test.to_numpy(), dtype=torch.float32)

y_train_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
y_val_t = torch.tensor(y_val.to_numpy(), dtype=torch.float32)
y_test_t = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

In [6]:
print(X_train.shape)
E_val.head()
print(E_train.to_numpy()[:5,:])
print(type(E_train.to_numpy()[:5,:]))

(4900, 1)
[[1427.53099622 1579.03161942 1114.79041062]
 [1426.04151013 1449.64630092 1088.78747916]
 [1264.56889781 1254.08520076  983.22585311]
 [1234.26522067 1242.45619537  996.4668084 ]
 [1288.74137211 1465.28530247 1015.43333954]]
<class 'numpy.ndarray'>


In [7]:
import torch.nn as nn

import torch.optim as optim

class MoeGatingNetwork(nn.Module):
    def __init__(self, input_dim: int, n_experts: int = 3):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(input_dim, 3),
            nn.Tanh(),
            nn.Linear(3, n_experts)
        )

    def forward(self, x, expert_preds):
        # Inference
        logits = self.gate(x)                 # (batch, 3)

        # Softmax
        weights = torch.softmax(logits, dim=1)

        # Prediction
        y_hat = torch.sum(weights * expert_preds, dim=1, keepdim=True)

        return y_hat, weights

In [8]:
print(X_train_t.shape[1])

1


In [ ]:
# ---------------------------------------------------
# MODEL
# ---------------------------------------------------
model = MoeGatingNetwork(input_dim=X_train_t.shape[1], n_experts=3)

# ---------------------------------------------------
# LOSS
# ---------------------------------------------------
criterion = nn.MSELoss()
history_train_loss = []
history_val_loss = []

# ---------------------------------------------------
# OPTIMIZATION
# ---------------------------------------------------
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------------------------------
# LOOP PARAMETERS
# ---------------------------------------------------
n_epochs = 1000
best_val_loss = float("inf")
best_state = None
patience = 20
patience_counter = 0

for epoch in range(n_epochs):
    
    # ----------------------
    # TRAIN
    # ----------------------
    model.train()
    epoch_loss_train = 0

    # Step 1: Clear previous gradients
    optimizer.zero_grad()

    # Step 2: Prediction (Forward pass)
    y_pred_train, w_train = model(X_train_t, E_train_t)

    # Step 3: Compute Loss
    loss_train = criterion(y_pred_train, y_train_t)

    # Step 4: Backpropagation (Backward pass)
    loss_train.backward()

    # Step 5: Update weights
    optimizer.step()

    rmse_train = torch.sqrt(loss_train)
    history_train_loss.append(rmse_train.item())

    # ----------------------
    # VALIDATION
    # ----------------------    
    model.eval()
    epoch_loss_val = 0

    with torch.no_grad():

        # Step 1: Inference
        y_pred_val, w_val = model(X_val_t, E_val_t)

        # Step 2: Compute batch loss
        loss_val = criterion(y_pred_val, y_val_t)
        
        rmse_val = torch.sqrt(loss_val)
        history_val_loss.append(rmse_val.item())

    # ----------------------
    # EARLY STOPPING
    # ---------------------- 
    if loss_val.item() < best_val_loss:
        best_val_loss = loss_val.item()
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    # ----------------------
    # VISUALIZATION
    # ----------------------
    if epoch % 20 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={rmse_train.item():.4f} | "
            f"val_loss={rmse_val.item():.4f}"
        )

    #if patience_counter >= patience:
    #    print(f"Early stopping at epoch {epoch}")
    #    break

if best_state is not None:
    model.load_state_dict(best_state)

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    y_pred_val, weights = model(X_val_t, E_val_t)

y_true = y_val_t.cpu().numpy().flatten()
y_pred = y_pred_val.cpu().numpy().flatten()

plt.figure(figsize=(10, 5))
plt.plot(y_true, label="True", alpha=0.7)
plt.plot(y_pred, label="Pred", alpha=0.7)
plt.legend()
plt.title("True vs Predicted")
plt.xlabel("Sample")
plt.ylabel("Target")
plt.grid()
plt.show()

In [ ]:
def mae_relative(y_true, y_pred):
    # MAE = Mean Average Error
    mae = np.mean(np.abs(y_pred - y_true))
    mean_target = np.mean(y_true)
    return 100 * mae / mean_target

mae_rel = mae_relative(y_true, y_pred)
mae = np.mean(np.abs(y_pred - y_true))

In [ ]:

print(f"MAE: {mae}")
print(f"MAE Relative: {mae_rel}%")

In [ ]:
import matplotlib.pyplot as plt 
plt.figure(figsize=(12, 5))

# Plot 1: Learning Curve
plt.subplot(1, 2, 1)
plt.plot(history_train_loss, color='blue', label='Train Loss')
plt.title("Convergence History (Train)")
plt.xlabel("Epochs")
plt.ylabel("RMSE Loss")
plt.grid(True)

# Plot 2: Validation Curve
plt.subplot(1, 2, 2)
plt.plot(history_val_loss, color='blue', label='Train Loss')
plt.title("Convergence History (Validation)")
plt.xlabel("Epochs")
plt.ylabel("RMSE Loss")
plt.grid(True)

# 2. Predictions vs. Experts

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    y_pred_val, w_val  = model(X_val_t, E_val_t)

y_true = y_val_t.cpu().numpy()
y_pred = y_pred_val.cpu().numpy()
weights = w_val.cpu().numpy()

# randomforest
exp_1 = E_val.to_numpy()[:,0]
# lgbm
exp_2 = E_val.to_numpy()[:,1]
# elasticnet
exp_3 = E_val.to_numpy()[:,2]

# windnorm
wind_norm = X_test.to_numpy()


start, end = 400,500

err_moe  = np.abs(y_true - y_pred.squeeze())
err_rf   = np.abs(y_true - exp_1)
err_lgbm = np.abs(y_true - exp_2)
err_enet = np.abs(y_true - exp_3)

err_moe  = (y_true - y_pred.squeeze())
err_rf   = (y_true - exp_1)
err_lgbm = (y_true - exp_2)
err_enet = (y_true - exp_3)

# ----------------------------------
# COLOR MAP
# ----------------------------------
colors = {
    "true": "black",
    "moe": "tab:red",
    "rf": "tab:blue",
    "lgbm": "tab:orange",
    "enet": "tab:green",
    "wind": "tab:purple"
}

wind_norm = X_val["Wind_Norm"].to_numpy()

plt.figure(figsize=(15, 24))

# ----------------------------------
# 1 — True vs MoE
# ----------------------------------
plt.subplot(7, 1, 1)
plt.plot(y_true, label="True", color=colors["true"], alpha=0.8)
plt.plot(y_pred, label="MoE", color=colors["moe"], linewidth=2)
plt.legend()
plt.ylabel("Units in $(MW)$")
plt.title("Full validation series - MoE vs True")
plt.grid()

# ----------------------------------
# 2 — All models
# ----------------------------------
plt.subplot(7, 1, 2)
plt.plot(y_true, label="True", color=colors["true"], alpha=0.6)
plt.title("Full series (Experts comparison)")
plt.plot(y_pred, label="MoE", color=colors["moe"], linewidth=2)

plt.plot(exp_1, label="RF", color=colors["rf"], alpha=0.7)
plt.plot(exp_2, label="LGBM", color=colors["lgbm"], alpha=0.7)
plt.plot(exp_3, label="ENet", color=colors["enet"], alpha=0.7)

plt.legend()
plt.ylabel("Units in $(MW)$")
plt.title("Full validation series - Experts comparison")
plt.grid()

# ----------------------------------
# 3 — Zoom MoE
# ----------------------------------
plt.subplot(7, 1, 3)
plt.plot(y_true, label="True", color=colors["true"], alpha=0.8)
plt.plot(y_pred, label="MoE", color=colors["moe"], linewidth=2)
plt.legend()
plt.title(f"MoE vs True - Zoom ({start}–{end})")
plt.xlim((start, end))
plt.ylabel("Units in $(MW)$")
plt.grid()

# ----------------------------------
# 4 — Zoom all models
# ----------------------------------
plt.subplot(7, 1, 4)
plt.plot(y_true, label="True", color=colors["true"], alpha=0.6)
plt.plot(y_pred, label="MoE", color=colors["moe"], linewidth=2)

plt.plot(exp_1, label="RF", color=colors["rf"], alpha=0.7)
plt.plot(exp_2, label="LGBM", color=colors["lgbm"], alpha=0.7)
plt.plot(exp_3, label="ENet", color=colors["enet"], alpha=0.7)

plt.legend()
plt.title(f"Experts comparison - Zoom ({start}–{end})")
plt.xlim((start, end))
plt.ylabel("Units in $(MW)$")
plt.grid()

# ----------------------------------
# 5 — Weights
# ----------------------------------
plt.subplot(7, 1, 5)
plt.plot(weights[:, 0], label="RF", color=colors["rf"], alpha=0.9)
plt.plot(weights[:, 1], label="LGBM", color=colors["lgbm"], alpha=0.9)
plt.plot(weights[:, 2], label="ENet", color=colors["enet"], alpha=0.9)

plt.legend()
plt.title(f"Gating weights - Zoom ({start}–{end})")
plt.xlim((start, end))
plt.ylabel("Weight")
plt.grid()


# ----------------------------------
# 6 — Absolute errors
# ----------------------------------
plt.subplot(7, 1, 6)
plt.plot(err_rf, label="RF", color=colors["rf"], alpha=0.9)
plt.plot(err_lgbm, label="LGBM", color=colors["lgbm"], alpha=0.9)
plt.plot(err_enet, label="ENet", color=colors["enet"], alpha=0.9)
plt.plot(err_moe, label="MoE", color=colors["moe"], alpha=0.9, linestyle = 'dashed')

plt.legend()
plt.title(f"Error - Zoom ({start}–{end})")
plt.xlim((start, end))
plt.ylabel("|error| $(MW)$")
plt.grid()


# ----------------------------------
# 7 — Wind feature
# ----------------------------------
plt.subplot(7, 1, 7)
plt.plot(wind_norm, label="Wind_Norm", color=colors["wind"], alpha=0.9)

plt.legend()
plt.title(f"Wind_Norm (Zoom {start}–{end})")
plt.xlim((start, end))
plt.ylabel("Wind $(m/s)$")
plt.grid()

# ----------------------------------
# FINAL DISPLAY
# ----------------------------------
plt.tight_layout()
plt.show()

In [ ]:
start, end = 400, 600

err_moe  = np.abs(y_true - y_pred.squeeze())
err_rf   = np.abs(y_true - exp_1)
err_lgbm = np.abs(y_true - exp_2)
err_enet = np.abs(y_true - exp_3)


print(y_true.shape)
print(y_pred.squeeze().shape)

In [ ]:

plt.figure(figsize=(12, 5))
plt.plot(err_moe,  label="MoE",  color="tab:red",    linewidth=2)
plt.plot(err_rf,   label="RF",   color="tab:blue",   alpha=0.8)
plt.plot(err_lgbm, label="LGBM", color="tab:orange", alpha=0.8)
plt.plot(err_enet, label="ENet", color="tab:green",  alpha=0.8)

plt.title("Absolute error in zoom area (400–600)")
plt.xlabel("Sample")
plt.ylabel("|error|")
plt.legend()
plt.grid()
plt.xlim((start, end))
plt.tight_layout()
plt.show()